# Week 11 — Anomaly Detection for Sentinel Prime AI

Prototypes anomaly-detection workflows on **two real public benchmarks** (no InGen data):

| Dataset | What | Licence |
|---|---|---|
| **NAB** `realKnownCause/machine_temperature_system_failure` | 22,695 real machine-temperature readings with 4 labelled failure windows | MIT (Numenta) |
| **SKAB** `valve1` | multivariate 8-sensor testbed with per-point anomaly labels | GPL-3.0 (Skoltech) |

Five detectors on identical splits: Isolation Forest, One-Class SVM, LOF, AutoEncoder (all PyOD) and an EWMA control-chart baseline.

## 1. Load the real benchmarks (downloads + caches)

In [ ]:
import subprocess, sys
print(subprocess.run([sys.executable,'-m','src.anomaly.data_loader'],capture_output=True,text=True,cwd='../..').stdout)

## 2. Run all detectors + evaluation + adaptive-threshold study + operational frontier

In [ ]:
out=subprocess.run([sys.executable,'-m','src.anomaly.run_all'],capture_output=True,text=True,cwd='../..').stdout
print(out[-3000:])

## 3. Results table (precision / recall / F1 / AP / FAR / time-to-detection)

In [ ]:
import pandas as pd
pd.set_option('display.width',160)
res=pd.read_csv('../../data/week11/results.csv')
print(res[['dataset','model','precision','recall','f1','avg_precision','false_alarm_rate','ttd_seconds','events_detected','events_total']].to_string(index=False))

## 4. Licences (every dataset documented — success criterion)

In [ ]:
lic=pd.read_csv('../../data/week11/dataset_licenses.csv')
for _,r in lic.iterrows(): print(f"{r['dataset']}: {r['license']}\n  {r['url']}\n  {r['citation']}\n")

## 5. Adaptive vs fixed threshold

The naive rolling quantile cuts false alarms but collapses recall — NAB's windows are ~2 days long, so the rolling baseline absorbs the anomaly. Baseline-freezing confirms the diagnosis (recall recovers) but over-corrects. See the report for the verdict.

In [ ]:
print(pd.read_csv('../../data/week11/adaptive_threshold.csv')[['scheme','precision','recall','f1','false_alarm_rate','ttd_seconds']].to_string(index=False))

## 6. Operational frontier — the finding that matters

Point-level false-alarm rate is a misleading metric. At the **alert level** (one alert per contiguous run, which is what a product pages on), a persistence filter cuts alert load ~33× while recall *rises*.

In [ ]:
print(pd.read_csv('../../data/week11/operational_frontier.csv')[['persistence_readings','threshold','point_recall','events_caught','events_total','false_alerts_per_day']].to_string(index=False))

## 7. Charts + 2-page operational framing

In [ ]:
print(subprocess.run([sys.executable,'-m','src.anomaly.build_report'],capture_output=True,text=True,cwd='../..').stdout)

## 8. Takeaways
- **AutoEncoder** best on NAB (F1 0.555), **LOF** best on SKAB (F1 0.545); both beat the control-chart baseline.
- The 5% point-level false-alarm budget is **not reachable** at 80% recall — that's a model-quality ceiling (AP≈0.51), not a tuning problem. Reported honestly rather than hidden.
- **Persistence filtering is the win**: 120-min sustained rule → 0.50 false alerts/day (vs 16.35), catching 4/4 real failures at 88% recall.
- **Adaptive thresholds lost** on this data and the reason is instructive: sustained anomalies contaminate a rolling baseline. Recommend fixed + persistence, re-baselined on a schedule.
- Recommend **two tiers**: a fast tier for unambiguous intrusion, a sustained tier for degradation.